# 7b. The same model on an extra sample of the BNE sixteenth century

Notebook 6 adds 807 BNE records of 1501–1600 to the sample (`data/llm_bne16_extra_sample.tsv`), because that catalog and century
had only 193 records in it. Here I classify them with the same model, the same instructions and the same settings as notebook 7,
without the note on the author. I ran it in Google Colab with a T4 GPU; upload the file to `/content` first. Notebook 8 reads the
answers from `data/llm_bne16_extra_results.tsv`.

---

# 7b. El mismo modelo sobre una muestra extra de la BNE del siglo XVI

El notebook 6 suma a la muestra 807 registros BNE de 1501–1600 (`data/llm_bne16_extra_sample.tsv`), porque ese catálogo y ese
siglo tenían solo 193 registros en ella. Acá los clasifico con el mismo modelo, las mismas instrucciones y los mismos parámetros
que en el notebook 7, sin la nota sobre el autor. Lo corrí en Google Colab con una GPU T4; antes hay que subir el archivo a
`/content`. El notebook 8 lee las respuestas de `data/llm_bne16_extra_results.tsv`.

In [ ]:
# Celda 1: Paths and model
EXTRA_SAMPLE_FILE = "/content/llm_bne16_extra_sample.tsv"
EXTRA_RESULTS_FILE = "/content/llm_bne16_extra_results.tsv"
MODEL = "qwen3:8b"
MODELS_DIR = "/content/ollama_models"

In [ ]:
# Celda 2: Install Ollama (zstd is needed to unpack it)
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Celda 3: Start the Ollama server in its own session, so interrupting a cell does not stop it
import os, subprocess, time
os.environ["OLLAMA_MODELS"] = MODELS_DIR
server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
                          start_new_session=True)
time.sleep(5)
!ollama --version

In [ ]:
# Celda 4: Download the model (about 5 GB)
!ollama pull {MODEL}

In [ ]:
# Celda 5: Load the extra sample
import pandas as pd
extra = pd.read_csv(EXTRA_SAMPLE_FILE, sep="\t", dtype={"id": str}).fillna("")
extra["lexicon_class"].value_counts()

In [ ]:
# Celda 6: Instructions for the model, from data/labeling_criteria.md plus a non-religious category
INSTRUCTIONS = """Clasificás ediciones españolas de 1500-1800 a partir del título, el autor y, si la hay, una nota sobre el autor.
Elegí UNA categoría:
- espiritual: oración, meditación, ejercicios espirituales, devoción, novenas, rosarios, vía crucis, perfección, ascética y mística, desengaños, preparación para la muerte o la comunión, vida cristiana, consuelos, poesía devota (Granada, Kempis, Teresa, Nieremberg...).
- predicable: sermones, oraciones evangélicas, fúnebres o panegíricas, sermonarios, cuaresmas, pláticas.
- historia_sacra: vidas de santos y venerables, milagros, imágenes y santuarios, crónicas de órdenes, canonizaciones, historia eclesiástica.
- teologia_moral: casos de conciencia, sumas morales, confesionarios, instrucciones de confesores.
- catecismo: catecismos, doctrinas cristianas, cartillas.
- teologia: dogmática, controversia, tratados doctrinales.
- biblia: Biblias, salmos, comentarios bíblicos.
- liturgia: misales, breviarios, oficios, horas, ceremoniales, villancicos.
- gobierno_religioso: reglas, constituciones, sínodos, edictos, bulas, indulgencias, pleitos y memoriales de conventos, Inquisición.
- no_religioso: literatura, teatro, novela, derecho, historia civil, ciencia, gramática, política.
La nota sobre el autor ayuda cuando el título es poco claro, pero lo que decide es el título.
Si el título es ambiguo, elegí el uso principal que sugiere."""

CATEGORIES = ["espiritual", "predicable", "historia_sacra", "teologia_moral", "catecismo",
              "teologia", "biblia", "liturgia", "gobierno_religioso", "no_religioso"]

In [ ]:
# Celda 7: Ask the model for one category; the answer is forced to be one of CATEGORIES
import json, requests

def classify(author, title, author_bio):
    question = "Autor: " + author + "\nTítulo: " + title
    if author_bio:
        question = question + "\nNota sobre el autor: " + author_bio
    answer_format = {"type": "object",
                     "properties": {"category": {"type": "string", "enum": CATEGORIES}},
                     "required": ["category"]}
    reply = requests.post("http://localhost:11434/api/chat", json={
        "model": MODEL,
        "messages": [{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": question}],
        "format": answer_format,
        "think": False,
        "stream": False,
        "options": {"temperature": 0},
    })
    return json.loads(reply.json()["message"]["content"])["category"]

classify("Granada, Luis de", "Guia de pecadores", "")   # Should be "espiritual"

In [ ]:
# Celda 8: Classify the extra sample by title and author, saving every 100 records in case the session ends
start = time.time()
llm_categories = []
for i, row in extra.iterrows():
    llm_categories.append(classify(row["author"], row["title"], ""))
    if len(llm_categories) % 100 == 0:
        print(len(llm_categories), f"{time.time() - start:.0f} s")
        extra.head(len(llm_categories)).assign(llm_category=llm_categories)[["source", "id", "llm_category"]].to_csv(
            EXTRA_RESULTS_FILE, sep="\t", index=False)
extra["llm_category"] = llm_categories
extra[["source", "id", "llm_category"]].to_csv(EXTRA_RESULTS_FILE, sep="\t", index=False)
pd.crosstab(extra["lexicon_class"], extra["llm_category"])